# Model C: Transformers (from scratch and pretrained DistilBERT)

Textbook Ch 12 (self-attention, transformers). (1) a small transformer encoder trained from scratch; (2) pretrained DistilBERT (Hugging Face `distilbert-base-uncased`), fully fine-tuned; (3) DistilBERT with the encoder FROZEN (only the classification head trains).

**Setup:** Runtime > Change runtime type > GPU (T4). Run all cells top to bottom. Results are appended to `results/results_C.csv`; do NOT edit `common.py` (frozen shared code).

In [ ]:
%pip install -q datasets transformers
import os
if not os.path.exists("common.py"):
    REPO_URL = "https://github.com/ZhongVang/csci167-rotten-tomatoes.git"   # group repo (private until submission)
    !git clone $REPO_URL project
    %cd project

In [ ]:
# Optional (Colab): keep results on Drive so they survive session timeouts.
# Must run BEFORE `import common`.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ["RESULTS_DIR"] = "/content/drive/MyDrive/CSCI167_results"
except Exception:
    pass  # running locally: results go to ./results

In [ ]:
OWNER = "C"
RESULTS_FILE = f"results_{OWNER}.csv"
SEEDS = [0, 1, 2]

import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from common import *

print(save_env_info(OWNER))                  # versions + hardware for the report
data = load_data()
print({k: len(v[0]) for k, v in data.items()}, "-> train/val/test sizes")
print("label counts (train):", np.bincount(data["train"][1]))

## Models

In [ ]:
class SmallTransformer(nn.Module):
    """Encoder-only transformer with learned position embeddings (Ch 12), mean-pooled for classification."""
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=2, ff=256, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.pos = nn.Embedding(MAX_LEN, d_model)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, ff, dropout, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.head = nn.Linear(d_model, 2)

    def forward(self, ids, mask):
        x = self.emb(ids) + self.pos(torch.arange(ids.size(1), device=ids.device))
        x = self.enc(x, src_key_padding_mask=~mask)
        m = mask.unsqueeze(-1).float()
        return self.head((x * m).sum(1) / m.sum(1).clamp(min=1))

set_seed(0)
_, _, _, v = make_vocab_loaders(data, 64, 0)
print("SmallTransformer trainable / total:", count_params(SmallTransformer(len(v))))
print("DistilBERT fine-tuned trainable / total:", count_params(HFClassifier()))
print("DistilBERT frozen encoder trainable / total:", count_params(HFClassifier(freeze_encoder=True)))

## Runs: 3 seeds x training fractions (100%, 25%, 10%)

From-scratch model uses the shared protocol (Adam 1e-3, batch 64, 20 epochs, patience 5). DistilBERT uses lr 2e-5, batch 32, 4 epochs, patience 2 (standard fine-tuning budget; report this difference in pretraining/budget in the write-up). The frozen-head variant uses lr 1e-3, 10 epochs.

In [ ]:
def run_scratch(seed, frac):
    set_seed(seed)
    train_dl, val_dl, test_dl, vocab = make_vocab_loaders(data, 64, seed, train_frac=frac)
    model = SmallTransformer(len(vocab))
    return train_and_eval(model, (train_dl, val_dl, test_dl), run_name=f"tfm_f{frac}_s{seed}",
                          model_name="SmallTransformer(scratch)", seed=seed, optimizer="adam", lr=1e-3,
                          epochs=20, patience=5, batch_size=64, train_frac=frac, results_file=RESULTS_FILE)

def run_bert(seed, frac, freeze):
    set_seed(seed)
    loaders = make_hf_loaders(data, "distilbert-base-uncased", 32, seed, train_frac=frac)[:3]
    model = HFClassifier("distilbert-base-uncased", freeze_encoder=freeze)
    tag = "bertfrozen" if freeze else "bert"
    return train_and_eval(model, loaders, run_name=f"{tag}_f{frac}_s{seed}",
                          model_name="DistilBERT(frozen)" if freeze else "DistilBERT(finetuned)",
                          seed=seed, optimizer="adam", lr=1e-3 if freeze else 2e-5,
                          epochs=10 if freeze else 4, patience=2, batch_size=32, train_frac=frac,
                          extra={"pretrained": "distilbert-base-uncased", "frozen_encoder": freeze},
                          results_file=RESULTS_FILE, save_checkpoint=False)

for frac in [1.0, 0.25, 0.1]:
    for seed in SEEDS:
        run_scratch(seed, frac)
        run_bert(seed, frac, freeze=False)
        run_bert(seed, frac, freeze=True)

## Results: summary table, learning curves, error analysis

In [ ]:
summarize(f'results_{OWNER}.csv')   # mean +/- std over seeds, per configuration

In [ ]:
plot_history("bert_f1.0_s0", "DistilBERT (fine-tuned): train vs validation (seed 0)");

In [ ]:
import json
h = json.load(open(f"{RESULTS_DIR}/history/bert_f1.0_s0.json"))
preds, labels = np.array(h["test_preds"]), np.array(data["test"][1])
texts = data["test"][0]
cm = np.zeros((2, 2), int)
for y, p in zip(labels, preds): cm[y, p] += 1
print("confusion matrix (rows=true neg/pos, cols=pred neg/pos):\n", cm)
wrong = np.where(preds != labels)[0]
print(len(wrong), "errors. Examples (true, pred, text):")
for i in wrong[:20]: print(labels[i], preds[i], "|", texts[i])
# TODO for the report: what do the errors have in common? (sarcasm, negation, mixed reviews, very short text...)